In [62]:
import os
import geopandas as gpd
import folium
from folium import FeatureGroup
os.getcwd()

'c:\\Users\\User\\Schoolwork\\MSGIS\\Webmapping\\HVI_leaflet_map'

In [81]:
gdb_path = os.path.join(os.getcwd(), 'HeatVulnerabilityFinal.gdb')

gdf = gpd.read_file(gdb_path, layer='HeatVulnerability_Blocks')

gdf = gdf.to_crs(epsg=4326)


# Display the first few rows of the data
print(gdf.head())

# List all the column names
columns = gdf.columns.tolist()
print("Column names:", columns)

# Multiply all values in the SVI column by 10 to put it on an integer scale
if 'CDC_SVI' in gdf.columns:
    gdf['CDC_SVI'] = (gdf['CDC_SVI'] * 10).round(0).astype(int)  # Multiply by 10, round to nearest integer, and convert to int
    print("Updated CDC_SVI column values (scaled to integer):")
    print(gdf['CDC_SVI'].head())
else:
    print("Column 'CDC_SVI' not found in the dataset.")

#to help with visualization, to make it in integer scale
#columns 

# Check the geometry type (should be 'Polygon' if it's polygon data)
geometry_type = gdf.geometry.type.unique()
print("Geometry type(s) in the dataset:", geometry_type)

   Join_Count  TARGET_FID BLKGRP BLOCK            GEOID    Celsius        Fah  \
0           1           1      1  1024  370210002001024  28.996535  84.193763   
1           2           2      2  2019  370210002002019  33.004543  91.408177   
2           3           3      2  2020  370210002002020  32.610769  90.699385   
3           2           4      2  2040  370210006002040  32.802437  91.044386   
4           2           5      1  1010  370210006001010  24.235385  75.623694   

   CDC_SVI  InVEST  Evapotr  Albedo_block  Shape_Length    Shape_Area  \
0  0.64810   240.0      238         973.0      0.007181  8.946223e-07   
1  0.44335   199.0      229        1046.0      0.006146  1.289329e-06   
2  0.58080   192.0      228         971.0      0.005247  2.511943e-07   
3  0.36400   226.0      232         938.0      0.006723  3.727511e-07   
4  0.38330   850.0      245         693.0      0.027129  2.692798e-05   

                                            geometry  
0  MULTIPOLYGON (((

# Backup in case something goes wrong

In [2]:
import os
import geopandas as gpd
import folium
import json  # Import json to parse GeoJSON string
from folium import FeatureGroup
import branca

gdb_path = os.path.join(os.getcwd(), 'HeatVulnerabilityFinal.gdb') #replace this if needed


gdf = gpd.read_file(gdb_path)
    
    # Attempt to fix invalid geometries using a buffer with 0 distance
gdf['geometry'] = gdf['geometry'].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)

gdf = gdf.dropna(subset=['CDC_SVI', 'InVEST', 'Evapotr', 'Albedo_block', 'Celsius'])
    
    # Ensure the CRS is in EPSG:4326 for compatibility with the basemap
gdf = gdf.to_crs(epsg=4326)

# Multiply all values in the SVI column by 10 to put it on an integer scale
if 'CDC_SVI' in gdf.columns:
    gdf['CDC_SVI'] = (gdf['CDC_SVI'] * 10).round(0).astype(int) 
    print("Updated CDC_SVI column values (scaled to integer):")
    print(gdf['CDC_SVI'].head())
else:
    print("Column 'CDC_SVI' not found in the dataset.")

    
# Function to create a legend for each layer
def create_legend(map_object, colormap, layer_name, min_value, max_value):
    # Truncate numbers to 1 decimal place
    min_value = round(min_value, 1)
    max_value = round(max_value, 1)
    
    # Generate a stepped colormap with rounded thresholds
    colormap = colormap.to_step(n=10, round_method='int')  # Use 'int' to round to the nearest integer
    
    # Generate the HTML for the colormap
    colormap_html = colormap._repr_html_()
    
    # Create the legend HTML
    legend = folium.Element(f"""
    <div id="legend-{layer_name.replace(' ', '-')}" style="display: none; position: absolute; bottom: 20px; left: 20px; z-index: 9999; background-color: white; padding: 10px; border: 1px solid black;">
        <b>Legend for {layer_name}</b><br>
        {colormap_html}
        <div style="text-align: center;">{min_value} -> {max_value}</div>
    </div>
    """)
    map_object.get_root().html.add_child(legend)

# Function to read GDB and create the web map
def create_map():
   
    
    map_center = [35.5337, -82.5285]  # Change to your desired center coordinates
    m = folium.Map(location=map_center, zoom_start=12)

    # Add base layers (OpenStreetMap and White background)
    folium.TileLayer('CartoDB positron', name="Minimal Background", control=True).add_to(m)  # White background layer
    
    # Map original column names to display names
    columns_to_visualize = {
        'CDC_SVI': 'Social Vulnerability',
        'InVEST': 'Urban Cooling Capacity',
        'Evapotr': 'Raw Evapotranspiration',
        'Albedo_block': 'Raw Albedo',
        'Celsius': 'Degrees Celsius'
    }

    # Color gradients for each column
    color_schemes = {
        'CDC_SVI': branca.colormap.linear.YlOrRd_09.scale(gdf['CDC_SVI'].min(), gdf['CDC_SVI'].max()),  # Reverse the color scale
        'InVEST': branca.colormap.linear.Greens_09.scale(gdf['InVEST'].min(), gdf['InVEST'].max()),    # Light Green to Dark Green
        'Evapotr': branca.colormap.linear.Greens_09.scale(gdf['Evapotr'].min(), gdf['Evapotr'].max()),    # Light Green to Dark Green
        'Albedo_block': branca.colormap.linear.YlOrRd_09.scale(gdf['Albedo_block'].min(), gdf['Albedo_block'].max()),  # Albedo colors
        'Celsius': branca.colormap.linear.YlOrRd_09.scale(gdf['Celsius'].min(), gdf['Celsius'].max())  # Yellow to Dark Red
    }

    # Loop over the selected columns and add them as separate layers on the map
    layer_groups = {}  # To store layer groups for each column
    for column in columns_to_visualize:
        if column in gdf.columns:
            # Create a FeatureGroup for each column
            layer_group = FeatureGroup(name=columns_to_visualize[column])
            
            # Convert the GeoDataFrame for the selected column to GeoJSON format
            geojson_data = gdf[['geometry', column]].to_json()

            # Parse the GeoJSON string into a dictionary
            geojson_dict = json.loads(geojson_data)

            # Ensure the GeoJSON has features to add
            if geojson_dict['features']:
                # Add the layer with a gradient color based on the column's values

                # Descriptions for each index
                descriptions = {
                    'Social Vulnerability': 'Social Vulnerability measures the resilience of communities when confronted by external stresses. This value reflects the SVI made by the Center for Disease Control (CDC)',
                    'Urban Cooling Capacity': 'Urban Cooling Capacity measures the ability of urban areas to mitigate heat. This value was calculated by the InVEST urban cooling model, found here: https://naturalcapitalproject.stanford.edu/invest/urban-cooling',
                    'Raw Evapotranspiration': 'Raw Evapotranspiration represents the water loss through evaporation and plant transpiration. This value was calculated by ECOSTRESS Evapotranspiration PT-JPL Daily L3 Global 70 m',
                    'Raw Albedo': 'Raw Albedo measures the reflectivity of surfaces. This value was calculated using Landsat 8 and 9 data: https://yceo.yale.edu/how-convert-landsat-dns-albedo. Higher albedo can reflect energy, lowering temperatures.',
                    'Degrees Celsius': 'Degrees Celsius represents the temperature in Celsius. This value was taken from Landsat 8 and 9 summer temperature data.'
                }
                folium.GeoJson(
                    geojson_dict,
                    style_function=lambda feature, column=column: {
                        'fillColor': color_schemes[column](feature['properties'][column]),
                        'color': 'black',  # Border color
                        'weight': 0.1,     # Border thickness
                        'fillOpacity': 0.8 # Transparency
                    },
                    popup=folium.GeoJsonPopup(
                        fields=[column],
                        aliases=[f"{columns_to_visualize[column]}: {descriptions[columns_to_visualize[column]]}"],
                        localize=True,
                        labels=True,
                        style="font-size: 12px;"
                    ),
                    tooltip=folium.GeoJsonTooltip(
                        fields=[column],
                        aliases=[f"{columns_to_visualize[column]}:"],
                        localize=True,
                        labels=True,
                        sticky=True,
                        style="font-size: 12px;"
                    )
                ).add_to(layer_group)
                
                # Add the layer group to the map's layers dictionary
                layer_groups[column] = layer_group

                # Add the legend for the current layer
                create_legend(m, color_schemes[column], columns_to_visualize[column], gdf[column].min(), gdf[column].max())

    # Add all layers to the map (each layer can be toggled)
    for layer_group in layer_groups.values():
        layer_group.add_to(m)

    # Add radio-style layer control
    folium.LayerControl(collapsed=False).add_to(m)

    # Save the map to an HTML file
    m.save("webmap.html")
    print("Map saved as 'webmap.html'.")
    with open("webmap.html", "a") as f:
         with open("webmap.html", "a") as f:
                f.write("""
                <script>
                // Debounce function to limit how frequently the event handler fires
                function debounce(fn, delay) {
                    let timer = null;
                    return function(...args) {
                        clearTimeout(timer);
                        timer = setTimeout(() => {
                            fn.apply(this, args);
                        }, delay);
                    };
                }
                
                document.addEventListener('DOMContentLoaded', function() {
                    var layerControls = document.querySelectorAll('.leaflet-control-layers-selector');
                    var legends = document.querySelectorAll('[id^="legend-"]');

                    // Separate base layers and content layers
                    var baseLayers = [];
                    var contentLayers = [];

                    layerControls.forEach(function(control) {
                        if (control.closest('.leaflet-control-layers-base')) {
                            baseLayers.push(control);
                        } else {
                            contentLayers.push(control);
                        }
                    });

                    // Function to show the legend for the selected layer
                    function showLegend(layerName) {
                        legends.forEach(function(legend) {
                            legend.style.display = "none"; // Hide all legends
                        });
                        var legend = document.getElementById("legend-" + layerName.replace(/\s+/g, '-'));
                        if (legend) {
                            legend.style.display = "block"; // Show the legend for the selected layer
                        }
                    }

                    // Ensure only one content layer is selected initially and show its legend
                    contentLayers.forEach(function(control) {
                        var layerName = control.nextSibling.innerText.trim();
                        if (layerName === "Degrees Celsius") {
                            control.checked = true;
                            showLegend(layerName);
                        } else {
                            control.checked = false;
                        }
                    });

                    // Add debounced event listeners to content layer controls
                    contentLayers.forEach(function(control) {
                        control.addEventListener('change', debounce(function() {
                            var layerName = control.nextSibling.innerText.trim(); // Get the layer name

                            // Deselect all other content layers
                            contentLayers.forEach(function(otherControl) {
                                if (otherControl !== control) {
                                    otherControl.checked = false;
                                }
                            });

                            // Show the legend for the selected layer
                            if (control.checked) {
                                showLegend(layerName);
                            } else {
                                legends.forEach(function(legend) {
                                    legend.style.display = "none";
                                });
                            }
                        }, 200));  // Adjust the delay (200ms here) as needed
                    });

                    // Base layers event listener remains unchanged
                    baseLayers.forEach(function(control) {
                        control.addEventListener('change', function() {
                            // Base layers do not affect content layers' legends
                        });
                    });
                });
                </script>
                """)
create_map()


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:265: UserWarning: More than one layer found in 'HeatVulnerabilityFinal.gdb': 'HeatVulnerability_Blocks' (default), 'HeatVulnerability_BlockGroups'. Specify layer parameter to avoid this warning.
  result = read_func(


Updated CDC_SVI column values (scaled to integer):
0    6
1    4
2    6
3    4
4    4
Name: CDC_SVI, dtype: int64
Map saved as 'webmap.html'.


In [11]:
print(gdf.crs)

EPSG:4326


In [12]:
# Check if GeoJSON is valid by inspecting the first few characters
geojson_data = gdf[['geometry', 'CDC_SVI']].to_json()
print(geojson_data[:200])  # Print the first 200 characters for inspection

{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {"CDC_SVI": 0.6481}, "geometry": {"type": "MultiPolygon", "coordinates": [[[[-82.56745899999999, 35.592232001000


In [13]:
print(gdf[['CDC_SVI', 'InVEST', 'Evapotr', 'Albedo_block', 'Celsius']].head())

   CDC_SVI  InVEST  Evapotr  Albedo_block    Celsius
0  0.64810   240.0      238         973.0  28.996535
1  0.44335   199.0      229        1046.0  33.004543
2  0.58080   192.0      228         971.0  32.610769
3  0.36400   226.0      232         938.0  32.802437
4  0.38330   850.0      245         693.0  24.235385


In [10]:
# Check if the geometries in the GeoDataFrame are valid
invalid_geometries = gdf[~gdf.is_valid]  # ~ is used to negate the valid geometries

# Print the number of invalid geometries
print(f"Number of invalid geometries: {len(invalid_geometries)}")

# Optionally, you can print the invalid geometries to inspect them
print("Invalid geometries:", invalid_geometries)

Number of invalid geometries: 0
Invalid geometries: Empty GeoDataFrame
Columns: [Join_Count, TARGET_FID, BLKGRP, BLOCK, GEOID, Celsius, Fah, CDC_SVI, InVEST, Evapotr, Albedo_block, Shape_Length, Shape_Area, geometry]
Index: []


In [8]:
# Attempt to fix invalid geometries using a buffer with 0 distance
gdf['geometry'] = gdf['geometry'].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)

# Check the number of invalid geometries after the fix
print(f"Number of invalid geometries after buffer fix: {len(gdf[~gdf.is_valid])}")

Number of invalid geometries after buffer fix: 0
